In [2]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [7]:
print(assistant.rag('How do I run Ollama locally?'))

To run Ollama locally, follow these steps:

1. **Install Ollama**: Visit [https://ollama.com/download](https://ollama.com/download) and choose your operating system:
   - **macOS**: Download the `.pkg` and install it.
   - **Windows**: Download the `.msi` and install it.
   - **Linux**: Run the command `curl -fsSL https://ollama.com/install.sh | sh` in the terminal.

2. **Run Ollama**: Open a terminal and type `ollama run llama3`. This will download the LLaMA 3 model (~4GB), start the model locally, and open a chat-like interface where you can type questions.

3. **Test the Ollama local server**: Run the command `curl http://localhost:11434` to test the server. You should receive a response similar to `{"models": [...]}`.

4. **Install the Python client**: Install the Python client with `pip install ollama`.

5. **Run a minimal Python example**: Use the following Python code to test Ollama:
   ```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "

In [8]:
print(assistant.rag("How do I run Olama locally?"))

There is no information provided about running Olama locally. The given context does not contain any details or instructions on how to run Olama. If you need assistance with running Olama, please provide more context or information about Olama and its requirements.


In [9]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='llama-3.3-70b-versatile',
    input=messages,
)

response.output_text

"You'd like to join a course, however, I need a bit more information. Could you please provide more details about the course you're interested in, such as its name, subject, or platform it's being hosted on? That way, I can give you a more accurate answer about whether it's still possible to join."

In [10]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )
    

In [13]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        'required': ["query"],
        'additionalProperties': False
    }
}

In [15]:
response  =openai_client.responses.create(
    model='llama-3.3-70b-versatile',
    input=messages,
    tools=[search_tool]
)

In [16]:
len(response.output)

2

In [20]:
call = response.output[1]

In [21]:
call

ResponseFunctionToolCall(arguments='{"query":"joining the course deadline"}', call_id='skdj08y5h', name='search', type='function_call', id='skdj08y5h', namespace=None, status='completed')

In [22]:
import json

args = json.loads(call.arguments)
args

{'query': 'joining the course deadline'}

In [23]:
call.name

'search'

In [24]:
results = search(**args)

In [26]:
result_json = json.dumps(results, indent=2)

In [27]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}


In [28]:
messages.append(call)

In [29]:
messages.append(function_call_output)

In [30]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"joining the course deadline"}', call_id='skdj08y5h', name='search', type='function_call', id='skdj08y5h', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'skdj08y5h',
  'output': '[\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2025."\n  },\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "q

In [32]:
response = openai_client.responses.create(
    model='llama-3.3-70b-versatile',
    input=messages,
    tools=[search_tool]
)

In [33]:
print(response.output_text)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [34]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(819, 33)

In [37]:
def calculate_llama_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
    "input_cost": input_cost,
    "output_cost": output_cost,
    "total_cost": total_cost
    }

result = calculate_llama_price(819, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.00014265


In [39]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
    }
    

In [40]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords.
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "I just discovered the course. Can I join it?"

messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]


In [41]:
response = openai_client.responses.create(
    model='llama-3.3-70b-versatile',
    input=messages,
    tools=[search_tool]
)


In [42]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        pritn(item.content[0].text)

function_call: search {"query":"joining the course"}


In [43]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function.\nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords.\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='resp_01kv5f9pkden1tg5fr70pcap7a', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"joining the course"}', call_id='51fv3vbzs', name='search', type='function_call', id='51fv3vbzs', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': '51fv3vbzs',
  'output': '[\n  {\n 

In [48]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model ='llama-3.3-70b-versatile',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"joining the course"}
iteration #2...
ASSISTANT:
Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [62]:
def agent_loop(instructions, question, model='llama-3.3-70b-versatile') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [63]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords fro the user question as possible when making first requests.

Make multiple searches. First perform search, analyse the results
and then perform more searches.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "I just discovered the course. Can I join it?"

In [65]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"joining the course deadline"}
iteration #2...
ASSISTANT:
You can still join the course, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [66]:
result

'You can still join the course, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [67]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

The question has to be about the course or its logistics, offtopic questions
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"Queen Gambit"}
iteration #2...
ASSISTANT:
The Queen's Gambit is a popular chess opening that starts with the moves 1.d4 d5 2.c4. It is considered one of the oldest and most aggressive openings in chess, offering a pawn to Black in order to put pressure on the position and gain a strategic advantage. The Queen's Gambit is often used by players of all levels, from beginners to grandmasters, and is known for its complex and dynamic nature.

In recent years, the term "Queen's Gambit" has also become associated with a popular Netflix series called "The Queen's Gambit," which tells the story of a young orphan girl who becomes a chess prodigy in the 1960s. The show explores themes of chess, addiction, and personal struggle, and has helped to popularize the game of chess among a wider audience.


In [69]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [70]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [71]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [72]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [73]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [75]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [85]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=openai_client

)

In [86]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

AttributeError: 'OpenAI' object has no attribute 'send_request'

In [88]:
result = agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"run Olama locally"}
iteration #2...
ASSISTANT:
To run Olama locally, you can follow these steps:

1. Clone the Olama repository from GitHub using the command `git clone https://github.com/olama/olama.git`.
2. Navigate to the cloned repository using `cd olama`.
3. Install the required dependencies using `pip install -r requirements.txt`.
4. Run the Olama server using `python app.py`.
5. Open a web browser and navigate to `http://localhost:5000` to access the Olama interface.

Note: These steps assume that you have Python and Git installed on your system. If you don't have them installed, you can download and install them from the official Python and Git websites.


In [89]:
result.cost

AttributeError: 'str' object has no attribute 'cost'